# Large dataset generation

In [ ]:
import json
import os
import re
import time
import urllib.request
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime, UTC
from pathlib import Path

import numpy as np
import pandas as pd

from simglucose.actuator.pump import InsulinPump
from simglucose.controller.basal_bolus_ctrller import BBController
from simglucose.patient.t1dpatient import T1DPatient
from simglucose.sensor.cgm import CGMSensor
from simglucose.simulation.env import T1DSimEnv
from simglucose.simulation.scenario import CustomScenario
from simglucose.simulation.sim_engine import SimObj, sim
from py_replay_bg.py_replay_bg import ReplayBG

ROOT = Path.cwd()
OUT = ROOT / "generated-datasets"
HUPA_DIR = Path("/tmp/hupa-ucm-preprocessed")

SIM_N_PATIENTS = 28
SIM_DAYS = 43
SIM_START = datetime(2024, 1, 1)
SIM_SEED = 0
SIM_WORKERS = 8
REPLAY_BW = 70.0
REPLAY_SEED = 1
REPLAY_WORKERS = 4
REPLAY_PARALLELIZE = True
SKIP_EXISTING = True

SIM_DIR = OUT / "simglucose" / "patients"
REPLAY_DIR = OUT / "replaybg" / "patients"
REPLAY_MAP = OUT / "replaybg" / "map"
for p in (SIM_DIR, REPLAY_DIR, REPLAY_MAP):
    p.mkdir(parents=True, exist_ok=True)

np.random.seed(SIM_SEED)


def sim_patient_ids(n):
    ids = []
    for prefix in ("adolescent", "adult", "child"):
        ids.extend(f"{prefix}#{i:03d}" for i in range(1, 11))
    return ids[:n]


def fmt_seconds(sec):
    if sec < 60:
        return f"{sec:.1f}s"
    if sec < 3600:
        return f"{sec / 60:.1f}m"
    return f"{sec / 3600:.2f}h"


EXPECTED_SIM_ROWS = SIM_DAYS * 288


def sim_csv_ok(path):
    if not path.exists() or path.stat().st_size < 10_000:
        return False
    n = len(pd.read_csv(path, usecols=["time"]))
    return n >= EXPECTED_SIM_ROWS - 2


def run_one_sim(job):
    idx, name, out_csv_str, meals, sim_start_iso, sim_days = job
    import time
    from datetime import datetime, timedelta
    from pathlib import Path

    import pandas as pd
    from simglucose.actuator.pump import InsulinPump
    from simglucose.controller.basal_bolus_ctrller import BBController
    from simglucose.patient.t1dpatient import T1DPatient
    from simglucose.sensor.cgm import CGMSensor
    from simglucose.simulation.env import T1DSimEnv
    from simglucose.simulation.scenario import CustomScenario
    from simglucose.simulation.sim_engine import SimObj, sim

    out_csv = Path(out_csv_str)
    sim_start = datetime.fromisoformat(sim_start_iso)
    tmp = out_csv.parent.parent.parent / "tmp-sim" / name.replace("#", "_")
    tmp.mkdir(parents=True, exist_ok=True)

    controller = BBController(target=120)
    controller.quest["CR"] = controller.quest["CR"].astype(float)
    controller.quest.loc[controller.quest["Name"].str.startswith("adult"), "CR"] *= 0.65

    t0 = time.perf_counter()
    env = T1DSimEnv(
        T1DPatient.withName(name),
        CGMSensor.withName("Dexcom", seed=idx),
        InsulinPump.withName("Insulet"),
        CustomScenario(start_time=sim_start, scenario=meals),
    )
    raw = sim(
        SimObj(env, controller, timedelta(days=sim_days), animate=False, path=str(tmp)),
    )
    step_min = raw.index.to_series().diff().dt.total_seconds().dropna().median() / 60
    amounts = pd.DataFrame({
        "glucose": raw["CGM"],
        "insulin": raw["insulin"].fillna(0) * step_min,
        "carbs": raw["CHO"].fillna(0) * step_min,
    })
    binned = amounts.resample("5min").agg({"glucose": "mean", "insulin": "sum", "carbs": "sum"})
    binned = binned.dropna(subset=["glucose"]).reset_index()
    binned = binned.rename(columns={binned.columns[0]: "time"})
    binned["patient"] = name
    df = binned[["patient", "time", "glucose", "insulin", "carbs"]]
    df.to_csv(out_csv, index=False)
    elapsed = time.perf_counter() - t0
    return name, elapsed, len(df)


run_started = datetime.now(timezone.utc)
sim_stats = {}
replay_stats = {}
sim_wall_seconds = None
replay_wall_seconds = None


In [ ]:
HUPA_DIR.mkdir(parents=True, exist_ok=True)
hupa_files = sorted(HUPA_DIR.glob("HUPA*.csv"))
if len(hupa_files) < 1:
    api = (
        "https://data.mendeley.com/api/datasets/3hbcscwz44/files"
        "?folder_id=c89e0831-7d0a-4fbb-9cb2-04e41161513c"
    )
    req = urllib.request.Request(api, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        listing = json.load(resp)
    files = [x for x in listing if re.fullmatch(r"HUPA\d+P\.csv", x["filename"])]
    for item in files:
        target = HUPA_DIR / item["filename"]
        if target.exists() and target.stat().st_size == item["size"]:
            continue
        link = item["content_details"]["download_url"]
        dl = urllib.request.Request(link, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(dl, timeout=120) as resp:
            target.write_bytes(resp.read())
    hupa_files = sorted(HUPA_DIR.glob("HUPA*.csv"))

print(f"HUPA CSV files: {len(hupa_files)} in {HUPA_DIR}")

In [ ]:
day_meals = ((7, 55), (12, 80), (18, 75), (22, 25))
meals = [(d * 24 + h, g) for d in range(SIM_DAYS) for h, g in day_meals]
sim_start_iso = SIM_START.isoformat()
patient_ids = sim_patient_ids(SIM_N_PATIENTS)

t_all = time.perf_counter()
sim_parts = []
pending = []

for idx, name in enumerate(patient_ids, start=1):
    out_csv = SIM_DIR / f"{name.replace('#', '_')}.csv"
    if SKIP_EXISTING and sim_csv_ok(out_csv):
        df = pd.read_csv(out_csv, parse_dates=["time"])
        sim_stats[name] = {"seconds": 0, "skipped": True, "rows": len(df)}
        sim_parts.append(df)
        print(f"[sim] {name} skipped ({len(df)} rows)")
        continue
    pending.append((idx, name, str(out_csv), meals, sim_start_iso, SIM_DAYS))

workers = min(SIM_WORKERS, len(pending)) if pending else 0
if pending and workers <= 1:
    for job in pending:
        name, elapsed, nrows = run_one_sim(job)
        df = pd.read_csv(SIM_DIR / f"{name.replace('#', '_')}.csv", parse_dates=["time"])
        sim_stats[name] = {"seconds": elapsed, "skipped": False, "rows": nrows}
        sim_parts.append(df)
        print(f"[sim] {name} {nrows} rows in {fmt_seconds(elapsed)}")
elif pending:
    print(f"[sim] {len(pending)} patients, {workers} workers")
    with ProcessPoolExecutor(max_workers=workers) as pool:
        futs = {pool.submit(run_one_sim, job): job[1] for job in pending}
        for fut in as_completed(futs):
            name, elapsed, nrows = fut.result()
            df = pd.read_csv(SIM_DIR / f"{name.replace('#', '_')}.csv", parse_dates=["time"])
            sim_stats[name] = {"seconds": elapsed, "skipped": False, "rows": nrows}
            sim_parts.append(df)
            print(f"[sim] {name} {nrows} rows in {fmt_seconds(elapsed)}")

sim_df = pd.concat(sim_parts, ignore_index=True)
sim_df.to_csv(OUT / "simglucose" / "simglucose.csv", index=False)
sim_wall_seconds = time.perf_counter() - t_all
print(f"simglucose combined: {len(sim_df)} rows, wall {fmt_seconds(sim_wall_seconds)}")


In [ ]:
def eligible_days(path):
    source = pd.read_csv(path, sep=";")
    source["time"] = pd.to_datetime(source["time"])
    source = source.sort_values("time").drop_duplicates("time").set_index("time")
    days = []
    for day, block in source.groupby(source.index.floor("D")):
        grid = pd.date_range(day, periods=288, freq="5min")
        if block.index.min() > grid[0] or block.index.max() < grid[-1]:
            continue
        aligned = block.reindex(grid)
        if aligned["glucose"].isna().mean() > 0.10:
            continue
        if aligned["carb_input"].fillna(0).sum() <= 0 or aligned["bolus_volume_delivered"].fillna(0).sum() <= 0:
            continue
        days.append(aligned)
    return days


def as_replay(aligned):
    carbs_g = aligned["carb_input"].fillna(0).to_numpy() * 10
    data = pd.DataFrame({
        "t": aligned.index,
        "glucose": aligned["glucose"].to_numpy(),
        "cho": carbs_g / 5,
        "bolus": aligned["bolus_volume_delivered"].fillna(0).to_numpy() / 5,
        "basal": aligned["basal_rate"].fillna(0).to_numpy() / 5,
    })
    data["cho_label"] = np.where(data["cho"] > 0, "S", "")
    data["bolus_label"] = np.where(data["bolus"] > 0, "C", "")
    return data


def run_one_replay(job):
    path_str, out_csv_str, map_folder_str, bw, seed, parallelize = job
    import os
    import time
    from pathlib import Path

    import numpy as np
    import pandas as pd
    from py_replay_bg.py_replay_bg import ReplayBG

    def eligible_days_local(path):
        source = pd.read_csv(path, sep=";")
        source["time"] = pd.to_datetime(source["time"])
        source = source.sort_values("time").drop_duplicates("time").set_index("time")
        days = []
        for day, block in source.groupby(source.index.floor("D")):
            grid = pd.date_range(day, periods=288, freq="5min")
            if block.index.min() > grid[0] or block.index.max() < grid[-1]:
                continue
            aligned = block.reindex(grid)
            if aligned["glucose"].isna().mean() > 0.10:
                continue
            if aligned["carb_input"].fillna(0).sum() <= 0 or aligned["bolus_volume_delivered"].fillna(0).sum() <= 0:
                continue
            days.append(aligned)
        return days

    def as_replay_local(aligned):
        carbs_g = aligned["carb_input"].fillna(0).to_numpy() * 10
        data = pd.DataFrame({
            "t": aligned.index,
            "glucose": aligned["glucose"].to_numpy(),
            "cho": carbs_g / 5,
            "bolus": aligned["bolus_volume_delivered"].fillna(0).to_numpy() / 5,
            "basal": aligned["basal_rate"].fillna(0).to_numpy() / 5,
        })
        data["cho_label"] = np.where(data["cho"] > 0, "S", "")
        data["bolus_label"] = np.where(data["bolus"] > 0, "C", "")
        return data

    path = Path(path_str)
    pid = path.stem
    out_csv = Path(out_csv_str)
    eligible = eligible_days_local(path)
    if not eligible:
        return pid, {"error": "no eligible days", "rows": 0}

    t0 = time.perf_counter()
    os.makedirs(map_folder_str, exist_ok=True)
    twin = ReplayBG(
        blueprint="multi-meal",
        save_folder=map_folder_str,
        yts=5,
        exercise=False,
        seed=seed,
        verbose=False,
        plot_mode=False,
    )
    fit = eligible[0]
    twin.twin(
        data=as_replay_local(fit),
        bw=bw,
        save_name=pid.lower(),
        twinning_method="map",
        parallelize=parallelize,
    )
    t_fit = time.perf_counter() - t0

    pieces = []
    for aligned in eligible:
        data = as_replay_local(aligned)
        result = twin.replay(data=data, bw=bw, save_name=pid.lower(), twinning_method="map")
        cgm = np.asarray(result["cgm"]["median"], dtype=float)
        pieces.append(pd.DataFrame({
            "patient": pid,
            "time": data["t"].to_numpy(),
            "glucose": cgm,
            "insulin": data["basal"].to_numpy() * 5 + data["bolus"].to_numpy() * 5,
            "carbs": data["cho"].to_numpy() * 5,
        }))

    df = pd.concat(pieces, ignore_index=True)
    df.to_csv(out_csv, index=False)
    elapsed = time.perf_counter() - t0
    return pid, {
        "seconds": elapsed,
        "fit_seconds": t_fit,
        "skipped": False,
        "rows": len(df),
        "eligible_days": len(eligible),
    }


t_all = time.perf_counter()
replay_parts = []
replay_pending = []

for path in hupa_files:
    pid = path.stem
    out_csv = REPLAY_DIR / f"{pid}.csv"
    if SKIP_EXISTING and out_csv.exists() and out_csv.stat().st_size > 10_000:
        df = pd.read_csv(out_csv, parse_dates=["time"])
        replay_stats[pid] = {"seconds": 0, "skipped": True, "rows": len(df)}
        replay_parts.append(df)
        print(f"[replay] {pid} skipped ({len(df)} rows)")
        continue
    replay_pending.append(
        (
            str(path),
            str(out_csv),
            str(REPLAY_MAP / pid),
            REPLAY_BW,
            REPLAY_SEED,
            REPLAY_PARALLELIZE,
        )
    )

rep_workers = min(REPLAY_WORKERS, len(replay_pending)) if replay_pending else 0
if replay_pending and rep_workers <= 1:
    for job in replay_pending:
        pid, st = run_one_replay(job)
        if st.get("error"):
            replay_stats[pid] = {**st, "seconds": 0, "skipped": True}
            print(f"[replay] {pid} {st['error']}")
            continue
        df = pd.read_csv(REPLAY_DIR / f"{pid}.csv", parse_dates=["time"])
        replay_stats[pid] = st
        replay_parts.append(df)
        print(
            f"[replay] {pid} {st['rows']} rows, {st['eligible_days']} days, "
            f"{fmt_seconds(st['seconds'])} (MAP {fmt_seconds(st['fit_seconds'])})"
        )
elif replay_pending:
    print(f"[replay] {len(replay_pending)} patients, {rep_workers} workers")
    with ProcessPoolExecutor(max_workers=rep_workers) as pool:
        futs = {pool.submit(run_one_replay, job): Path(job[0]).stem for job in replay_pending}
        for fut in as_completed(futs):
            pid, st = fut.result()
            if st.get("error"):
                replay_stats[pid] = {**st, "seconds": 0, "skipped": True}
                print(f"[replay] {pid} {st['error']}")
                continue
            df = pd.read_csv(REPLAY_DIR / f"{pid}.csv", parse_dates=["time"])
            replay_stats[pid] = st
            replay_parts.append(df)
            print(
                f"[replay] {pid} {st['rows']} rows, {st['eligible_days']} days, "
                f"{fmt_seconds(st['seconds'])} (MAP {fmt_seconds(st['fit_seconds'])})"
            )

if replay_parts:
    replay_df = pd.concat(replay_parts, ignore_index=True)
    replay_df.to_csv(OUT / "replaybg" / "replaybg.csv", index=False)
else:
    replay_df = pd.DataFrame()

replay_wall_seconds = time.perf_counter() - t_all
print(f"replaybg combined: {len(replay_df)} rows, wall {fmt_seconds(replay_wall_seconds)}")


In [ ]:
run_finished = datetime.now(UTC)
print("GENERATION SUMMARY")
print(f"started_utc:  {run_started.isoformat(timespec='seconds')}")
print(f"finished_utc: {run_finished.isoformat(timespec='seconds')}")
print("config:")
print(f"  SIM_N_PATIENTS={SIM_N_PATIENTS}  SIM_DAYS={SIM_DAYS}  SIM_START={SIM_START.isoformat()}")
print(f"  SIM_SEED={SIM_SEED}  SIM_WORKERS={SIM_WORKERS}  REPLAY_BW={REPLAY_BW}  REPLAY_SEED={REPLAY_SEED}")
print(f"  REPLAY_WORKERS={REPLAY_WORKERS}  REPLAY_PARALLELIZE={REPLAY_PARALLELIZE}  SKIP_EXISTING={SKIP_EXISTING}")
print(f"  OUT={OUT}  HUPA_DIR={HUPA_DIR}")

print("\nsimglucose patients:")
for name, st in sim_stats.items():
    if st.get("skipped"):
        print(f"  {name}: skipped, rows={st['rows']}")
    else:
        print(f"  {name}: {fmt_seconds(st['seconds'])}, rows={st['rows']}")
sim_done = [st for st in sim_stats.values() if not st.get("skipped")]
if sim_wall_seconds is not None:
    print(f"simglucose wall: {fmt_seconds(sim_wall_seconds)}")
if sim_done:
    mean_sim = sum(st["seconds"] for st in sim_done) / len(sim_done)
    print(f"simglucose mean per patient (generated): {fmt_seconds(mean_sim)}")
print(f"simglucose combined rows: {len(sim_df)}, patients: {sim_df['patient'].nunique() if len(sim_df) else 0}")

print("\nreplaybg patients:")
for pid, st in replay_stats.items():
    if st.get("error"):
        print(f"  {pid}: {st['error']}")
    elif st.get("skipped"):
        print(f"  {pid}: skipped, rows={st['rows']}")
    else:
        fit_s = fmt_seconds(st.get("fit_seconds", 0))
        print(f"  {pid}: {fmt_seconds(st['seconds'])}, MAP {fit_s}, days={st['eligible_days']}, rows={st['rows']}")
rep_done = [st for st in replay_stats.values() if not st.get("skipped") and st.get("rows", 0) > 0]
if replay_wall_seconds is not None:
    print(f"replaybg wall: {fmt_seconds(replay_wall_seconds)}")
if rep_done:
    mean_rep = sum(st["seconds"] for st in rep_done) / len(rep_done)
    print(f"replaybg mean per patient (generated): {fmt_seconds(mean_rep)}")
print(f"replaybg combined rows: {len(replay_df)}, patients: {replay_df['patient'].nunique() if len(replay_df) else 0}")

total = (sim_wall_seconds or 0) + (replay_wall_seconds or 0)
print(f"\ntotal wall time this run: {fmt_seconds(total)}")
print(f"outputs: {OUT}")
